# Manufacturing – Predictive Maintenance & Equipment Health
# Microsoft Fabric RTI – Setup Guide

---

## Business Context

Unplanned equipment downtime costs manufacturing plants **~$260k/hour** on average (up to **$3M/hour** in automotive). This demo streams live sensor telemetry from 4 machines through Microsoft Fabric Real-Time Intelligence to detect anomalies, predict failures, and alert maintenance teams — all in real time.

**Who consumes the output:** Maintenance engineers, reliability managers, production supervisors, plant managers.

## Architecture

```
IoT Sensors → Eventstream → BronzeEquipmentTelemetry
                                    │
                              Update Policy
                                    │
                           SilverEquipmentTelemetry
                                    │
                  ┌─────────────────┼─────────────────┐
          mv_MachineHealth    mv_AnomalyCounts    mv_SensorTrends
                  │                  │                   │
            Dashboard          Data Activator       Dashboard
```

## What You'll Build

| Fabric Item | Name |
|---|---|
| Eventhouse | `PredictiveMaintenanceMonitoring_EH` |
| KQL Database | `PredictiveMaintenanceMonitoring_EH` (auto-created) |
| Eventstream | `EquipmentTelemetryStream_ES` |
| Custom Endpoint | `EquipmentTelemetrySource` |
| Generator Notebook | `PredictiveMaintenance_Generator` |
| Real-Time Dashboard | `PredictiveMaintenance_Dashboard` |
| Data Activator | `PredictiveMaintenanceAlert` |

## Checklist
- [ ] Step 1 – Create Workspace
- [ ] Step 2 – Create Eventhouse
- [ ] Step 3 – Enable OneLake Availability
- [ ] Step 4 – Create Eventstream + Custom Endpoint
- [ ] Step 5 – Upload & Run Generator Notebook
- [ ] Step 6 – Define Eventstream Topology → Bronze Table
- [ ] Step 7 – Run KQL Schema Script
- [ ] Step 8 – Build Real-Time Dashboard
- [ ] Step 9 – Configure Data Activator Alert
- [ ] Step 10 – Git Integration
- [ ] Step 11 – Deployment Pipeline

## STEP 1 – Log in & Create Workspace

1. Go to [app.fabric.microsoft.com](https://app.fabric.microsoft.com)
2. Sign in with your Microsoft 365 / Fabric account
3. In the left nav, click **Workspaces** → **+ New workspace**
4. Name: `PredictiveMaintenance_RTI_Demo`
5. Under **License mode**, select **Fabric capacity** (F2+ or Trial)
6. Click **Apply**

## STEP 2 – Create Eventhouse

1. Inside the workspace, click **+ New item** → search **Eventhouse**
2. Name: `PredictiveMaintenanceMonitoring_EH`
3. Click **Create**
4. A KQL Database with the same name is auto-created inside the Eventhouse

## STEP 3 – Enable OneLake Availability

1. Open the KQL Database (`PredictiveMaintenanceMonitoring_EH`)
2. Click the **pencil/toggle** next to **OneLake availability** → set to **Active**
3. This makes KQL data available to other Fabric workloads (Lakehouse, notebooks, etc.)

> ⚠️ With OneLake enabled, `.clear table` commands are blocked. Toggle off temporarily if you need to clear data.

## STEP 4 – Create Eventstream with Custom Endpoint

1. In the workspace, click **+ New item** → search **Eventstream**
2. Name: `EquipmentTelemetryStream_ES`
3. Click **Create**
4. In the Eventstream editor, click **New source** → **Custom endpoint**
5. Name the endpoint: `EquipmentTelemetrySource`
6. Click **Add**
7. Click the **Keys** tab on the custom endpoint
8. **Copy** the following (you'll paste them into the generator notebook):
   - **Event Hub Name** (e.g. `es_xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx`)
   - **Connection String – Primary Key** (starts with `Endpoint=sb://...`)

> 🔑 Keep these credentials handy — you'll paste them in Step 5.

## STEP 5 – Upload & Run Generator Notebook

1. In the workspace, click **+ New item** → **Import notebook**
2. Upload `PredictiveMaintenance_Generator.ipynb`
3. Open the notebook
4. **Cell 1** — Run to install packages (`azure-eventhub`, `faker`)
5. **Cell 2** — Paste your Event Hub Name and Connection String from Step 4
6. Run **Cell 2** to load configuration
7. Run **Cell 3** to load machine definitions
8. Run **Cell 4** to load generator functions
9. Run **Cell 5** to start background streaming → look for `🚀 Generator running in background`

> 💡 Leave the notebook open. The generator runs as a daemon thread while the kernel is alive.
>
> ⏱️ Increase session timeout: **Run → Configure session → 60-120 min**

## STEP 6 – Define Eventstream Topology → Bronze Table

1. Go back to `EquipmentTelemetryStream_ES`
2. You should see events flowing into the Custom Endpoint (check the preview)
3. Click **New destination** → **Eventhouse**
4. Select your Eventhouse: `PredictiveMaintenanceMonitoring_EH`
5. Select KQL Database: `PredictiveMaintenanceMonitoring_EH`
6. Table name: **Create new** → `BronzeEquipmentTelemetry`
7. Input data format: **JSON**
8. Click **Add** → **Publish**
9. Wait ~30 seconds, then verify in KQL:

```kql
BronzeEquipmentTelemetry | count
```

You should see a growing count.

## STEP 7 – Run KQL Schema Script

Open the KQL Database and run each block from `predictivemaintenance_createAll.kql` **one at a time**.

### ⚠️ IMPORTANT: Run blocks individually!
Copy-paste one block → Run → Wait for completion → Next block.
Running multiple `.create`/`.alter` commands at once causes a `Recognition Error`.

### Block execution order:

| Block | Command | What it creates |
|---|---|---|
| 1 | `.create table SilverEquipmentTelemetry` | Silver table with typed columns |
| 2 | `.create-or-alter function SeverityFromHealth` | Health score → label mapper |
| 3 | `.create-or-alter function TransformEquipmentTelemetry` | Bronze → Silver transformation |
| 4 | `.alter table ... policy update` | Update Policy (auto-fires on insert) |
| 5 | `.create materialized-view mv_MachineHealth` | Hourly health averages (Gold) |
| 6 | `.create materialized-view mv_AnomalyCounts` | Hourly anomaly counts (Gold) |
| 7 | `.create materialized-view mv_SensorTrends` | 15-min sensor trends (Gold) |
| 8 | `.create-or-alter function ActiveAlerts` | Live alerts convenience function |
| 9 | `.create-or-alter function MachineStatusBoard` | Latest status per machine |

### Verify Silver is populating:
```kql
SilverEquipmentTelemetry | count
```

> 💡 Silver only populates on **new** Bronze inserts. If count is 0, wait for the next generator batch (~3 seconds).

## STEP 8 – Build Real-Time Dashboard

1. In the workspace, click **+ New item** → **Real-Time Dashboard**
2. Name: `PredictiveMaintenance_Dashboard`
3. Connect to data source: your KQL Database (`PredictiveMaintenanceMonitoring_EH`)
4. Set **auto-refresh** to **30 seconds**
5. Add tiles using the queries below

---

### Tile 1 — Machine Health Status Board (Stat Cards)
```kql
MachineStatusBoard()
| project machineName, healthScore, severityLabel, sensorType, sensorValue, unit, isAnomaly
```
**Visual:** Stat / Multi-stat cards  
**Formatting:** Green (≥80), Yellow (40–79), Red (<40)

---

### Tile 2 — Live Sensor Trends (Hero Tile)
```kql
SilverEquipmentTelemetry
| where timestamp between (_startTime .. _endTime)
| where sensorType == "Temperature"  // change per sensor type
| summarize avgValue = round(avg(sensorValue), 1) by machineName, bin(timestamp, 5m)
| render timechart
```
**Visual:** Line chart  
**X-axis:** time, **Y-axis:** avgValue, **Series:** machineName  
💡 *This is the most important tile — crisis events show as dramatic spikes.*

---

### Tile 3 — Anomaly Counter (Last Hour)
```kql
SilverEquipmentTelemetry
| where timestamp between (_startTime .. _endTime)
| where isAnomaly == true
| summarize anomalyCount = count() by machineName
| order by anomalyCount desc
```
**Visual:** Bar chart (horizontal)  
**Color:** Red (#E74C3C)

---

### Tile 4 — Active Alerts Table
```kql
ActiveAlerts()
| project machineName, location, alertCount, worstHealth, sensors, latestAlert
| order by alertCount desc
```
**Visual:** Table  
💡 *This populates when anomalies/crises fire — may be empty during calm periods.*

---

### Tile 5 — Anomalies by Sensor Type
```kql
materialized_view('mv_AnomalyCounts')
| where timestamp between (_startTime .. _endTime)
| summarize totalAnomalies = sum(anomalyCount) by sensorType
| order by totalAnomalies desc
```
**Visual:** Pie chart or Donut  
**Colors:** Temperature=Red, Vibration=Orange, Pressure=Blue, Current=Yellow

---

### Tile 6 — Health Score Trend Over Time
```kql
materialized_view('mv_MachineHealth')
| where timestamp between (_startTime .. _endTime)
| project machineName, timestamp, avgHealthScore
| render timechart
```
**Visual:** Area chart  
**X-axis:** time, **Y-axis:** avgHealthScore, **Series:** machineName  
Add reference lines at 40 (Poor) and 80 (Good)

---

### Tile 7 — Sensor Value Heatmap
```kql
materialized_view('mv_SensorTrends')
| where timestamp between (_startTime .. _endTime)
| project machineName, sensorType, timestamp, avgValue
| extend label = strcat(machineName, " – ", sensorType)
```
**Visual:** Table (with conditional formatting on avgValue)  
Or use a multi-line time chart filtered to one machine at a time.

---

### Tile 8 — Maintenance KPI Summary
```kql
let total_events = toscalar(SilverEquipmentTelemetry | where timestamp >= ago(24h) | count);
let anomaly_events = toscalar(SilverEquipmentTelemetry | where timestamp >= ago(24h) | where isAnomaly == true | count);
let machines_warning = toscalar(MachineStatusBoard() | where severityLabel in ("Critical", "Poor") | count);
print 
    TotalReadings_24h = total_events,
    Anomalies_24h = anomaly_events,
    AnomalyRate = round(100.0 * anomaly_events / total_events, 1),
    MachinesInWarning = machines_warning
```
**Visual:** Stat / KPI cards  
Shows overall plant health at a glance.

## STEP 9 – Configure Data Activator Alert

1. On the **Active Alerts Table** tile (Tile 4), click **⋯** → **Set alert**
2. Configure:

| Field | Value |
|---|---|
| Check | On each event grouped by |
| Grouping field | `machineName` |
| When | `alertCount` |
| Condition | Becomes greater than |
| Value | `5` |
| Action | **Message me in Teams** |

3. Name: `PredictiveMaintenanceAlert`
4. Click **Create**

### Recommended Alert Thresholds

| Alert | Condition | Justification |
|---|---|---|
| High Temperature | Temperature > 85°C | Motor insulation degradation risk; fire hazard |
| Vibration Anomaly | Vibration > 3× baseline | Bearing wear or imbalance — imminent failure |
| Pressure Drop | Pressure loss > 30% | Hydraulic leak — equipment damage + defective parts |
| Recurring Stops | ≥5 stops/hour | Underlying mechanical or sensor issue |
| Power Surge | Current draw +20% above normal | Electrical strain — motor/fuse damage risk |

### Advanced Actions (mention in demo)
- **Power Automate**: Create a maintenance ticket in ServiceNow or SAP PM
- **Teams Adaptive Card**: Rich notification with machine details + recommended action
- **API Webhook**: Trigger external CMMS maintenance workflow

## STEP 10 – Connect Fabric Workspace to GitHub

1. In the workspace, go to **Settings** → **Git integration**
2. Select **GitHub** as the provider
3. Authenticate with your GitHub account
4. Select your repository and branch (e.g., `main`)
5. Set the folder to `/` (root)
6. Click **Connect and sync**

> 💡 Fabric syncs items by **name** — ensure notebook names in Fabric match filenames in Git.
>
> ⚠️ Edits in VS Code to root `.ipynb` files may not sync to Fabric if the item uses Fabric's folder structure. Safest flow: edit in Fabric → commit to Git.

## STEP 11 – Set Up Deployment Pipeline (Dev → Prod)

1. In Fabric, go to **Deployment Pipelines** → **+ New pipeline**
2. Name: `PredictiveMaintenance_Pipeline`
3. Assign your workspace to the **Development** stage
4. Create (or assign) a **Production** workspace to the Production stage
5. Click **Deploy** to promote items from Dev → Prod

> See `CICD_Patterns_Reference.md` for advanced CI/CD patterns including `fabric-cicd` Python package.

## 🎤 Demo Flow Recommendation

### Talking Points by Tile

1. **Machine Health Board** — "Here's the real-time health of all 4 machines. Green means healthy, red means action needed."
2. **Sensor Trends** — "This is our hero chart. Watch the temperature lines — when a crisis hits, you'll see the spike instantly."
3. **Anomaly Counter** — "This shows how many anomalies we've detected per machine in the current window."
4. **Active Alerts** — "Any machine that crosses our thresholds appears here automatically."
5. **Anomalies by Type** — "Temperature and vibration anomalies are the most common predictors of failure."
6. **Health Trend** — "Notice how the health score degrades before a failure — that's the predictive power."

### Crisis Demo
1. Show normal baseline (~10-15 min of streaming)
2. Go to generator notebook → **Cell 6** → Set `TRIGGER_TARGET = "CNC-007"` → Run
3. Switch to dashboard → watch temperature spike + health score drop in real time
4. Show the Data Activator alert arriving in Teams
5. Key message: *"From sensor spike to Teams alert in under 30 seconds — that's the power of Fabric RTI."*

### Macro Event Demo
1. In Cell 6, change `TRIGGER_TYPE = "macro"` and `TRIGGER_TARGET = "spare_parts_shortage"`
2. Show degraded readings across CNC-007 and Press Line 3
3. Key message: *"When supply chains disrupt, predictive maintenance becomes even more critical — we can't afford failures if parts aren't available."*

## 🛑 Stop the Generator

When the demo is over:
1. In the generator notebook, run **Cell 7** (or create a new cell with):
```python
GENERATOR_STATE['running'] = False
```
2. Optionally clear data in KQL:
```kql
.clear table BronzeEquipmentTelemetry data
.clear table SilverEquipmentTelemetry data
```
3. **Pause Fabric capacity** in Azure Portal to stop billing

## 🔧 Troubleshooting

| Problem | Fix |
|---|---|
| Bronze table empty | Check generator is running; check Eventstream is Active |
| Silver table empty | Update Policy only fires on new ingest; wait for next Bronze insert |
| Anomaly view empty | Wait for crisis burst or check anomaly condition in Silver |
| Dashboard tiles empty | Set time range to Last 1 hour; check data source connection |
| Kernel timeout | Re-run all cells; increase session timeout in Run → Configure session |
| Eventstream inactive after capacity resume | Open Eventstream → Activate destination |
| `.clear table` blocked | Toggle OneLake availability OFF, clear, toggle back ON |
| Connection string error | Re-copy from Eventstream Keys tab — may rotate on capacity pause/resume |

## ⏱️ Pre-Demo Checklist

| # | Action | Time |
|---|---|---|
| 1 | Resume Fabric capacity | 2 min |
| 2 | Open generator notebook → Run Cells 1-5 | 3 min |
| 3 | Wait for `🚀 Generator running` message | 10 sec |
| 4 | Verify: `BronzeEquipmentTelemetry \| count` is growing | 30 sec |
| 5 | Open dashboard → confirm tiles updating | 30 sec |
| 6 | Wait 10-15 min for baseline data | 15 min |
| 7 | Cell 6 ready for live crisis trigger | — |

**Total lead time: ~20 min before demo**